# Notebook 05 — Link MedMNIST Images to MPI

**Goal:** Assign each MedMNIST image a `patient_id` from the MPI.

**Linking logic:**
- Each image belongs to a folder (AbdomenCT, BreastMRI, ChestCT, CXR, Hand, HeadCT)
- Each patient in the MPI has a `medmnist_folder` assigned in Notebook 01
- Images are assigned to patients whose `medmnist_folder` matches the image folder
- BreastMRI images are only assigned to female patients

**Output:** `data_preparation/linked/imaging_linked.csv`

## 1. Imports & Paths

In [9]:
import pandas as pd
import numpy as np
import random
import os

MEDMNIST_DIR = "../raw/medmnist/"
MPI_PATH     = "../linked/patients_master.csv"
OUTPUT_DIR   = "../linked/"

FOLDERS = ["AbdomenCT", "BreastMRI", "ChestCT", "CXR", "Hand", "HeadCT"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 2. Load MPI

In [10]:
mpi = pd.read_csv(MPI_PATH)

print(f"MPI patients : {len(mpi)}")
print()
print("MedMNIST folder distribution in MPI:")
print(mpi["medmnist_folder"].value_counts())

MPI patients : 1163

MedMNIST folder distribution in MPI:
medmnist_folder
CXR          343
Hand         216
AbdomenCT    175
ChestCT      173
HeadCT       171
BreastMRI     85
Name: count, dtype: int64


## 3. Scan All Image Files

In [11]:
records = []

for folder in FOLDERS:
    folder_path = os.path.join(MEDMNIST_DIR, folder)
    if not os.path.exists(folder_path):
        print(f"WARNING: folder not found → {folder_path}")
        continue

    image_files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for img_file in image_files:
        records.append({
            "image_file"   : img_file,
            "image_path"   : os.path.join(folder, img_file),
            "modality"     : folder,
        })

    print(f"{folder:<15} → {len(image_files)} images")

images_df = pd.DataFrame(records)
print()
print(f"Total images found : {len(images_df)}")

AbdomenCT       → 10000 images
BreastMRI       → 8954 images
ChestCT         → 10000 images
CXR             → 10000 images
Hand            → 10000 images
HeadCT          → 10000 images

Total images found : 58954


## 4. Build Modality → Patient Pool Lookup

In [12]:
# Group patients by their assigned medmnist_folder
modality_patient_pools = mpi.groupby("medmnist_folder")["patient_id"].apply(list).to_dict()
all_patient_ids        = mpi["patient_id"].tolist()

print("Patient pool sizes per modality:")
for folder, pool in modality_patient_pools.items():
    print(f"  {folder:<15} → {len(pool)} patients")

Patient pool sizes per modality:
  AbdomenCT       → 175 patients
  BreastMRI       → 85 patients
  CXR             → 343 patients
  ChestCT         → 173 patients
  Hand            → 216 patients
  HeadCT          → 171 patients


## 5. Link Each Image to a Patient ID

In [13]:
def find_patient_for_image(modality):
    """
    Assign a patient_id to a medical image based on modality.
    Returns (patient_id, match_type)
    """
    if modality in modality_patient_pools:
        pool = modality_patient_pools[modality]
        if len(pool) > 0:
            return random.choice(pool), "modality_match"

    # Fallback
    return random.choice(all_patient_ids), "random_fallback"


print("Linking images to patient IDs...")
results = images_df["modality"].apply(find_patient_for_image)

images_df["patient_id"] = results.apply(lambda x: x[0])
images_df["match_type"] = results.apply(lambda x: x[1])

print("Done.")
print()
print("Match type distribution:")
print(images_df["match_type"].value_counts())

Linking images to patient IDs...
Done.

Match type distribution:
match_type
modality_match    58954
Name: count, dtype: int64


## 6. Reorder Columns

In [14]:
cols = ["patient_id", "match_type", "modality", "image_file", "image_path"]
images_df = images_df[cols]

print("Final columns:")
print(images_df.columns.tolist())
print()
images_df.head(5)

Final columns:
['patient_id', 'match_type', 'modality', 'image_file', 'image_path']



,patient_id,match_type,modality,image_file,image_path
0,23add813-11a9-abf7-f511-e3a7400ae578,modality_match,AbdomenCT,000000.jpeg,AbdomenCT\000000.jpeg
1,873049ed-d86d-74b2-84c4-cbc49b4384d0,modality_match,AbdomenCT,000001.jpeg,AbdomenCT\000001.jpeg
2,22b01ffb-7537-e873-3a7c-0d6961c498d7,modality_match,AbdomenCT,000002.jpeg,AbdomenCT\000002.jpeg
3,3d60584c-574a-16e0-ef8f-c8a7315878e9,modality_match,AbdomenCT,000003.jpeg,AbdomenCT\000003.jpeg
4,c49a00b1-cead-b093-f02c-be9b7789c584,modality_match,AbdomenCT,000004.jpeg,AbdomenCT\000004.jpeg


## 7. Quality Check

In [15]:
print("=== Imaging Linking Quality Check ===")
print(f"Total images             : {len(images_df)}")
print(f"Unique patients assigned : {images_df['patient_id'].nunique()}")
print(f"Patients with 0 images   : {len(mpi) - images_df['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(images_df["match_type"].value_counts())
print()
print("Images per modality:")
print(images_df["modality"].value_counts())
print()
print("Images per patient (stats):")
print(images_df.groupby("patient_id").size().describe())
print()
print("BreastMRI assigned to male patients (must be 0):")
breast_mri = images_df[images_df["modality"] == "BreastMRI"]
male_patients = mpi[mpi["GENDER"] == "M"]["patient_id"].tolist()
print(breast_mri[breast_mri["patient_id"].isin(male_patients)].shape[0])

=== Imaging Linking Quality Check ===
Total images             : 58954
Unique patients assigned : 1163
Patients with 0 images   : 0

Match type distribution:
match_type
modality_match    58954
Name: count, dtype: int64

Images per modality:
modality
AbdomenCT    10000
ChestCT      10000
CXR          10000
Hand         10000
HeadCT       10000
BreastMRI     8954
Name: count, dtype: int64

Images per patient (stats):
count    1163.000000
mean       50.691316
std        20.731506
min        16.000000
25%        34.000000
50%        50.000000
75%        60.000000
max       129.000000
dtype: float64

BreastMRI assigned to male patients (must be 0):
0


## 8. Save Output

In [16]:
output_path = os.path.join(OUTPUT_DIR, "imaging_linked.csv")
images_df.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {images_df.shape}")

Saved → ../linked/imaging_linked.csv
Shape  : (58954, 5)
